<a href="https://colab.research.google.com/github/COMP3608-Group-12/Project/blob/Fixing-NaN-values-issue/Group_12_Dataset_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Initial test commit

## STEP 1: Importing Packages

In [1]:
#importing packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import imblearn

## STEP 2: Load & Understand Data

In [5]:
#loading csv file into pandas dataframe
df = pd.read_csv('creditcard.csv')

#printing dataset information (how many rows, columns etc)
print(df.info())
print('----------------------------------------------------------')

#checking if any values are missing
print(df.isnull().sum())
print('----------------------------------------------------------')

# Show ONLY columns with missing values
print("Columns with missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print('----------------------------------------------------------')

#Dropping rows with missing values
df = df.dropna()

#Verifying that those rows have been dropped and there are no missing values
print("Total missing values after dropping:")
print(df.isnull().sum().sum())  # should be 0
print('----------------------------------------------------------')


#previewing the data
print(df.head(5))
print('----------------------------------------------------------')

#checking how many transactions are fraudulent and how many are non-fraudulent
print(df['Class'].value_counts())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45646 entries, 0 to 45645
Data columns (total 31 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Time    45646 non-null  int64  
 1   V1      45646 non-null  float64
 2   V2      45646 non-null  float64
 3   V3      45646 non-null  float64
 4   V4      45646 non-null  float64
 5   V5      45646 non-null  float64
 6   V6      45645 non-null  float64
 7   V7      45645 non-null  float64
 8   V8      45645 non-null  float64
 9   V9      45645 non-null  float64
 10  V10     45645 non-null  float64
 11  V11     45645 non-null  float64
 12  V12     45645 non-null  float64
 13  V13     45645 non-null  float64
 14  V14     45645 non-null  float64
 15  V15     45645 non-null  float64
 16  V16     45645 non-null  float64
 17  V17     45645 non-null  float64
 18  V18     45645 non-null  float64
 19  V19     45645 non-null  float64
 20  V20     45645 non-null  float64
 21  V21     45645 non-null  float64
 22

## Step 3: Split Data into Features & Target

In [6]:
#Splitting features and target
X = df.drop('Class', axis=1)
y = df['Class']

##Step 4: Decision Tree Model using KFold Cross Validation

In [7]:
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

#Decision Tree Model
model_dt = DecisionTreeClassifier(random_state=42)

# StratifiedKFold
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Store results
all_results = []

# Manual KFold Loop
for fold, (train_index, test_index) in enumerate(kf.split(X, y), 1):

    print(f"\n================== Fold {fold} ==================")

    # Split
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # SMOTE on training data only
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

    # Train model
    model_dt.fit(X_train_resampled, y_train_resampled)

    # Predict on test data
    y_pred = model_dt.predict(X_test)
    y_prob = model_dt.predict_proba(X_test)[:, 1]

    #Performance metrics
    results = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    }

    results_df = pd.DataFrame({
        "Metric": list(results.keys()),
        "Value": [round(value, 4) for value in results.values()]
    })

    print(results_df)

    print(f"\n============================================")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print(f"\n============================================")
    print("Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))

    # Appending results
    all_results.append(results)

# Taking average of all results
results_df = pd.DataFrame(all_results)

avg_results = results_df.mean()

final_df = pd.DataFrame({
    "Metric": avg_results.index,
    "Average Value": [round(val, 4) for val in avg_results.values]
})

print("\n================== AVERAGE PERFORMANCE ==================")
print(final_df)


================== Fold 1 ==================
      Metric   Value
0   Accuracy  0.9981
1  Precision  0.6571
2     Recall  0.8214
3   F1 Score  0.7302
4    ROC-AUC  0.9101

Confusion Matrix:
[[9089   12]
 [   5   23]]

Classification Report:
              precision    recall  f1-score   support

         0.0     0.9995    0.9987    0.9991      9101
         1.0     0.6571    0.8214    0.7302        28

    accuracy                         0.9981      9129
   macro avg     0.8283    0.9101    0.8646      9129
weighted avg     0.9984    0.9981    0.9982      9129


================== Fold 2 ==================
      Metric   Value
0   Accuracy  0.9981
1  Precision  0.7037
2     Recall  0.6786
3   F1 Score  0.6909
4    ROC-AUC  0.8388

Confusion Matrix:
[[9093    8]
 [   9   19]]

Classification Report:
              precision    recall  f1-score   support

         0.0     0.9990    0.9991    0.9991      9101
         1.0     0.7037    0.6786    0.6909        28

    accuracy             

## Step 6: Deep Neural Network

## Step 7: Logistic Regression with K Fold Cross Validation